<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/main/notebooks/01-processamento_pln.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Célula 1: Montagem do Google Drive para persistência de dados e acesso aos scripts SQL
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Célula 2: Instalação das bibliotecas, configuração de caminhos e criação/população do banco de dados
# 1. Instalação das bibliotecas (BERTimbau e BERTopic)
!pip install -q transformers torch pandas bertopic pysentimiento spacy
!python -m spacy download pt_core_news_lg

import pandas as pd
import sqlite3
import os
import torch

# 2. Configuração de caminhos
# Ajuste o caminho do DRIVE_DIR para onde estão os arquivos .sql
DRIVE_DIR = '/content/drive/MyDrive/mba-engsof-tcc/versao_final'
DB_FILE_NAME = 'base-dados.db'
DB_PATH = os.path.join(DRIVE_DIR, DB_FILE_NAME)

# Novos artefatos separados
SCHEMA_SQL = os.path.join(DRIVE_DIR, '01-schema.sql')
SEED_SQL = os.path.join(DRIVE_DIR, '02-seed_data.sql')

# 3. Função para inicializar o banco de dados
def inicializar_banco(db_path, schema_path, seed_path):
    print(f"🛠️ Inicializando banco de dados em: {db_path}")

    # Remove o banco anterior se desejar começar do zero (Opcional)
    # if os.path.exists(db_path): os.remove(db_path)

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        # Passo 1: Executa o Schema (Estrutura)
        print("📐 Aplicando estrutura (01-schema.sql)...")
        with open(schema_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())

        # Passo 2: Executa o Seed (Dados)
        print("🌱 Semeando dados (02-seed_data.sql)... Isso pode levar alguns segundos.")
        with open(seed_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())

        conn.commit()
        print("✅ Banco de dados pronto para o processamento!")

    except Exception as e:
        print(f"❌ Erro ao inicializar banco: {e}")
    finally:
        conn.close()

# 4. Execução e Hardware
device = 0 if torch.cuda.is_available() else -1

# Executa a inicialização
inicializar_banco(DB_PATH, SCHEMA_SQL, SEED_SQL)

print(f"\n✅ Ambiente configurado.")
if device == 0:
    print("🚀 GPU detectada! A análise será rápida.")
else:
    print("⚠️ GPU NÃO detectada. A análise do BERTimbau será lenta (CPU).")

In [ ]:
# Célula 3: Limpeza Estrutural e Filtro de Densidade (Antidoto ao Ruído Nominal)
import spacy
import sqlite3
import pandas as pd
import re

# Carrega o modelo de português
try:
    nlp = spacy.load("pt_core_news_lg")
except:
    !python -m spacy download pt_core_news_lg
    nlp = spacy.load("pt_core_news_lg")

def limpar_texto_estrutural(texto):
    if not texto or len(texto.strip()) < 3: return "RUIDO_CURTO"

    doc = nlp(texto)

    # Filtramos tokens válidos (substantivos, verbos, adjetivos e nomes próprios)
    # Ignoramos stop words e pontuação
    tokens = [t for t in doc if not t.is_stop and not t.is_punct and t.pos_ in ['NOUN', 'VERB', 'ADJ', 'PROPN']]

    if not tokens: return "RUIDO_VAZIO"

    # Métrica 1: Densidade de Nomes Próprios (PROPN)
    # Se mais de 70% do conteúdo significativo forem nomes próprios, é provavelmente uma lista/genealogia
    propn_count = len([t for t in tokens if t.pos_ == 'PROPN'])
    propn_ratio = propn_count / len(tokens)

    # Métrica 2: Presença de Ação/Estado
    # Antídotos existenciais raramente são frases sem verbos ou adjetivos
    has_action_or_state = any(t.pos_ in ['VERB', 'ADJ'] for t in tokens)

    # CASO CRÍTICO (Ex: Nesias e Hatifa):
    # Verso curto, alta proporção de nomes próprios e sem verbo
    if len(tokens) <= 3 and propn_ratio > 0.5 and not has_action_or_state:
        return "RUIDO_NOMINAL"

    # Retornamos o texto limpo (usando o texto original para preservar a semântica)
    return " ".join([t.text.lower() for t in tokens])

# Execução e Persistência
conn = sqlite3.connect(DB_PATH)
df_versos = pd.read_sql_query("SELECT id, texto FROM verso", conn)

print("🧼 Limpando textos e aplicando filtros de densidade gramatical...")
df_versos['texto_limpo'] = df_versos['texto'].apply(limpar_texto_estrutural)

cursor = conn.cursor()
cursor.execute("DELETE FROM verso_limpo")
df_versos[['id', 'texto_limpo']].rename(columns={'id': 'verso_id'}).to_sql(
    'verso_limpo',
    conn,
    if_exists='append', # Importante: 'append' preserva a estrutura e índices
    index=False
)

conn.commit()
conn.close()
print("✅ Célula 3 concluída! Ruídos nominais e estruturais pré-identificados.")

In [ ]:
# Célula 4: Classificação por Eixos Existenciais
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from transformers import pipeline
import sqlite3
import pandas as pd
import numpy as np
import os

# 1. Configuração e Carga de Dados
conn = sqlite3.connect(DB_PATH)
query = """
    SELECT v.id as verso_id, v.texto, l.abreviacao, g.id as genero_id, vl.texto_limpo
    FROM verso v
    JOIN verso_limpo vl ON v.id = vl.verso_id
    JOIN livro l ON l.id = v.livro_id
    JOIN genero_literario g ON g.id = l.genero_id
"""
df_input = pd.read_sql_query(query, conn)

# 2. Inicialização do Modelo BERTimbau (Zero-Shot)
# O uso de device=0 ativa a GPU no Google Colab para processamento em larga escala
embedding_model = pipeline("feature-extraction", model="neuralmind/bert-base-portuguese-cased", device=0)

descricoes_eixos = [
    # Eixo 0: Han (Cansaço vs. Refrigério)
    "Fadiga e Descanso da Alma: Sentimentos subjetivos de exaustão emocional, cansaço mental, peso da existência, prostração, desânimo, desfalecimento e o fardo de carregar a vida, em contraste com a busca por alívio espiritual, refrigério, paz interior, consolo e a restauração das forças vitais.",

    # Eixo 1: Bauman (Incerteza vs. Rocha)
    "Transitoriedade e Solidez: A angústia da impermanência humana, a sensação de que tudo é passageiro e instável, o medo da perda de fundamentos e a fragilidade dos laços, confrontados pela solidez absoluta de Deus, a rocha eterna, a fidelidade que não muda e o que permanece para sempre.",

    # Eixo 2: Frankl (Vazio vs. Sentido)
    "Vazio e Propósito de Vida: A crise de sentido, o sentimento de inutilidade, o sofrimento sem explicação, a falta de direção e o vácuo existencial, em oposição à descoberta da vocação, o valor intrínseco da vida humana, o chamado divino e a razão para continuar existindo.",

    # Eixo 3: Controle (Narrativo/Normativo)
    "Registros Históricos e Normas: Listas de nomes, genealogias de famílias, mortes de reis e sucessões políticas, contagem de anos, leis civis, rituais de sacrifício com animais, medidas de construção de templos e diálogos informativos sobre viagens ou logística."
]

model_topic = BERTopic(
    embedding_model=embedding_model,
    zeroshot_topic_list=descricoes_eixos,
    zeroshot_min_similarity=0.1,
    calculate_probabilities=True,
    vectorizer_model=CountVectorizer(ngram_range=(1, 2))
)

print("🤖 Classificando via Zero-Shot (Incluso: Poéticos, Proféticos, Evangelhos, Epístolas e Apocalíptico)...")
topics, probs_matrix = model_topic.fit_transform(df_input['texto_limpo'].tolist())

# 3. Lógica de Decisão e Veto Metodológico Híbrido
final_topics = []
final_probs = []

# Gêneros com "Passe Livre" para análise subjetiva profunda
# 3:Poético, 4:Profético, 5:Evangelho, 6:Epístola, 7:Apocalíptico
generos_subjetivos = [3, 4, 5, 6, 7]

for i, row in df_input.iterrows():
    # Extração das probabilidades (BERTimbau)
    p_esgotamento, p_transitoriedade, p_insignificancia, p_narrativo = probs_matrix[i][0:4]
    scores_existenciais = [p_esgotamento, p_transitoriedade, p_insignificancia]

    best_idx = np.argmax(scores_existenciais)
    best_score = scores_existenciais[best_idx]

    # --- Filtro de Qualidade de Dados (Veto) ---
    if row['genero_id'] not in generos_subjetivos:
        # Rigor máximo para Pentateuco (1) e Históricos (2)
        if p_narrativo > 0.30 or (p_narrativo > best_score * 0.85):
            final_topics.append(3) # Classificado como 'Outros'
            final_probs.append(p_narrativo)
            continue
    else:
        # Flexibilidade para Gêneros Existenciais:
        # Só veta se o tom narrativo for desproporcionalmente superior ao existencial
        if p_narrativo > 0.55 and p_narrativo > (best_score * 2.0):
            final_topics.append(3)
            final_probs.append(p_narrativo)
            continue

    final_topics.append(best_idx)
    final_probs.append(best_score)

# 4. Persistência no SQLite
print("💾 Atualizando banco de dados com a nova estrutura...")
mapa_eixos = {0: "Esgotamento (Han)", 1: "Transitoriedade (Bauman)", 2: "Insignificância (Frankl)", 3: "Outros"}
df_verso_topico = pd.DataFrame({'verso_id': df_input['verso_id'], 'topico_id': final_topics, 'similaridade': final_probs})

cursor = conn.cursor()
# Limpeza para evitar duplicidade ou dados órfãos da execução anterior
cursor.execute("DELETE FROM verso_topico")
cursor.execute("DELETE FROM topico")

# Inserção do Dicionário de Tópicos
df_mapa = pd.DataFrame([{'id': k, 'antidoto_referencia': v} for k, v in mapa_eixos.items()])
df_mapa.to_sql('topico', conn, if_exists='append', index=False)

# Inserção dos Resultados da Classificação
df_verso_topico.to_sql('verso_topico', conn, if_exists='append', index=False)

conn.commit()
conn.close()
print(f"✨ Célula 4 concluída! {len(df_verso_topico)} versículos processados.")

In [ ]:
# Célula 5: Análise de Sentimento Contextual e Cruzamento Existencial
from pysentimiento import create_analyzer
import pandas as pd
import sqlite3
from tqdm.auto import tqdm

# 1. Inicializar o Analisador
print("🚀 Carregando modelo Transformer para Sentimento (PT-BR)...")
# O analisador 'sentiment' para 'pt' é baseado em BERTimbau, ideal para o seu TCC
analyzer = create_analyzer(task="sentiment", lang="pt")

# 2. Busca do texto original e dos tópicos (Corrigido para 'verso')
conn = sqlite3.connect(DB_PATH)
df_input = pd.read_sql_query("""
    SELECT v.id as verso_id, v.texto, vt.topico_id
    FROM verso v
    JOIN verso_topico vt ON v.id = vt.verso_id
""", conn)

textos = df_input['texto'].tolist()
verso_ids = df_input['verso_id'].tolist()

# 3. Execução da análise em lotes (Aproveitando a GPU se disponível)
print(f"📊 Analisando carga emocional de {len(textos)} versículos (Incluindo novos gêneros)...")
sentimentos = []
batch_size = 64
mapa_num = {'POS': 1, 'NEU': 0, 'NEG': -1}

# O predict em lote é significativamente mais rápido no Colab
for i in tqdm(range(0, len(textos), batch_size)):
    lote = textos[i:i + batch_size]
    ids_lote = verso_ids[i:i + batch_size]
    preds_lote = analyzer.predict(lote)

    for idx, p in enumerate(preds_lote):
        # Capturamos as probabilidades brutas para análises de incerteza se necessário
        sentimentos.append({
            'verso_id': ids_lote[idx],
            'label': p.output,
            'sentimento_num': mapa_num.get(p.output, 0),
            'score_pos': p.probas.get('POS', 0),
            'score_neg': p.probas.get('NEG', 0),
            'score_neu': p.probas.get('NEU', 0)
        })

df_sent = pd.DataFrame(sentimentos)

# 4. Persistência dos Resultados
try:
    cursor = conn.cursor()
    # Limpamos para garantir que a nova classificação da Célula 4 seja a única presente
    cursor.execute("DELETE FROM verso_sentimento")

    # Inserimos os novos resultados (Integridade referencial com 'verso_id')
    df_sent.to_sql('verso_sentimento', conn, if_exists='append', index=False)
    conn.commit()
    print("\n✅ Sentimentos processados e salvos com sucesso.")

    # 5. RESULTADO FINAL: O DIAGNÓSTICO (PROBLEMA) VS. A CURA (ANTÍDOTO)
    print("\n📈 RESUMO EXECUTIVO: PROBLEMÁTICA (CRISE) VS. ANTÍDOTO (CURA)")

    res_final = pd.read_sql_query("""
        SELECT
            t.antidoto_referencia as Eixo_Filosofico,
            COUNT(*) as Total_Versos,
            SUM(CASE WHEN vs.sentimento_num = 1 THEN 1 ELSE 0 END) as Antidotos_Cura,
            SUM(CASE WHEN vs.sentimento_num = -1 THEN 1 ELSE 0 END) as Problematica_Crise,
            ROUND(AVG(vs.sentimento_num), 3) as Polaridade_Media
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        WHERE t.id != 3 -- Foco nos eixos Han, Bauman e Frankl
        GROUP BY t.antidoto_referencia
        ORDER BY Polaridade_Media DESC
    """, conn)

    # Exibe a tabela formatada no Colab
    display(res_final)

except Exception as e:
    print(f"❌ Erro na persistência: {e}")
finally:
    conn.close()